# YuE2 - UI (Colab)

Lokaldeki kurulumun Colab sürümü: YuE2 model sunucusu + Gradio arayüzü (**✦ Text to Song**: açıklama + söz → şarkı, **🎵 Cover**: MP3'ün melodisiyle yeni şarkı).

**Başlamadan:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4 GPU*** (yoksa A100). Hepsini birden çalıştırmak için *Çalışma zamanı → Tümünü çalıştır*.
Son hücre, siz **KAPAT** kutucuğunu işaretlemeden hiçbir şeyi kapatmaz. İş bitince kutucuğu işaretleyip son hücreyi çalıştırın; aksi halde GPU bağlı kaldıkça compute birimi harcanır.

## 1. GPU kontrolü

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
assert out.returncode == 0, "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU"
name, mem_mib, cap = [x.strip() for x in out.stdout.strip().splitlines()[0].split(",")]
GPU_MEM_GIB = int(mem_mib) / 1024
print(f"{name} · {GPU_MEM_GIB:.1f} GiB · compute {cap}")
if float(cap) < 8.0:
    print("⚠️ Bu GPU (ör. T4) BF16'yı donanımda desteklemiyor: çalışır ama yavaş olur; bellek tasarrufu açılacak.")

## 2. Google Drive (önbellek)
Colab her oturumda boş bir makineyle başlar; kalıcı olan tek yer Drive. `DRIVE_CACHE = True` iken model ağırlıkları (~15 GB) ve Python ortamları (~10 GB) **yalnızca ilk oturumda** indirilip kurulur ve Drive'daki `yue2_cache` klasörüne kaydedilir; sonraki oturumlarda oradan açılır. Drive bağlanırken Google bir izin penceresi açar.

Üretilen şarkılar da oturum kapanınca silinmesin istiyorsanız `USE_DRIVE = True` yapın.

In [ ]:
DRIVE_CACHE = True   # ağırlıkları ve Python ortamlarını Drive'da sakla: yalnızca ilk oturumda iner
USE_DRIVE = False    # şarkıları da Drive'a yaz
import os
OUTPUTS = "/content/yue2_outputs"
os.environ["YUE2_CACHE"] = ""
if DRIVE_CACHE or USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if DRIVE_CACHE:
    os.environ["YUE2_CACHE"] = "/content/drive/MyDrive/yue2_cache"
if USE_DRIVE:
    OUTPUTS = "/content/drive/MyDrive/yue2_outputs"
print("Önbellek:", os.environ["YUE2_CACHE"] or "kapalı (her oturumda indirilip kurulur)")
print("Çıktılar:", OUTPUTS)

In [ ]:
%%writefile /content/cache.sh
# Drive cache used by cells 4, 5 and 5b. Each part is one uncompressed tar (Drive is slow with many small files).
# The marker is the archive's last member: if a Drive upload was cut short, the marker is missing and we re-download.
cache_restore() {  # name path...
    local archive="$YUE2_CACHE/$1.tar" marker="/content/.cache_ok/$1"; shift
    [ -n "$YUE2_CACHE" ] && [ -f "$archive" ] || return 1
    echo "Drive önbelleğinden açılıyor: $archive ($(du -h "$archive" | cut -f1))"
    rm -f "$marker"
    tar -xf "$archive" -C / && [ -f "$marker" ] && return 0
    echo "⚠️ Önbellek eksik (Drive yüklemesi yarım kalmış olabilir); silinip yeniden indirilecek/kurulacak."
    rm -f "$archive"; rm -rf "$@"
    return 1
}
cache_save() {  # name path...
    local archive="$YUE2_CACHE/$1.tar" marker="/content/.cache_ok/$1"; shift
    [ -n "$YUE2_CACHE" ] || return 0
    echo "Drive önbelleğine yazılıyor (yalnızca bu oturumda): $archive"
    mkdir -p "$YUE2_CACHE" /content/.cache_ok && touch "$marker"
    local rel=(); for p in "$@" "$marker"; do rel+=("${p#/}"); done
    if tar -cf "$archive.part" -C / "${rel[@]}" && mv "$archive.part" "$archive"; then
        echo "Önbellek hazır: $(du -h "$archive" | cut -f1)"
    else
        rm -f "$archive.part"
        echo "⚠️ Önbelleğe yazılamadı (Drive dolu olabilir); sonraki oturumda yeniden indirilecek/kurulacak."
    fi
}
cache_prune() {  # prefix keep: delete the prefix's archives except keep
    [ -n "$YUE2_CACHE" ] || return 0
    for old in "$YUE2_CACHE/$1"*.tar; do
        [ "$old" = "$YUE2_CACHE/$2.tar" ] || rm -f "$old"
    done
}

## 3. Paketi yükle
`yue2-colab.zip` dosyasını seçin. Drive'ın ana klasörüne koyduysanız (ve 2. adımda Drive'ı bağladıysanız) oradan otomatik alınır.

In [ ]:
import os, shutil, zipfile
BUNDLE = "/content/yue2-colab.zip"
DRIVE_BUNDLE = "/content/drive/MyDrive/yue2-colab.zip"
if not os.path.exists(BUNDLE):
    if os.path.exists(DRIVE_BUNDLE):
        shutil.copy(DRIVE_BUNDLE, BUNDLE)
    else:
        from google.colab import files
        uploaded = files.upload()
        os.replace(next(iter(uploaded)), BUNDLE)
zipfile.ZipFile(BUNDLE).extractall("/content")
print(sorted(os.listdir("/content/yue-ui")))

## 4. Paketleri kur (ilk oturumda ~2-5 dk)
yue2'nin sabitlediği sürümler (torch 2.10, transformers 4.57.6, numpy 2.2.6) Colab'ın kendi paketlerinden ayrı bir ortama (`/content/yue2_ortam`) kurulur; Colab'ın paketleriyle çakışma olmaz. Önbellek açıksa ortam Drive'a yazılır ve sonraki oturumlarda kurulmadan oradan açılır; aşağıdaki sürümler ya da zip'teki yue2 wheel'i değişirse otomatik olarak yeniden kurulur. Son satırda **Whisper pipeline OK** görünmeli.

In [ ]:
%%bash
set -e
source /content/cache.sh
VENV=/content/yue2_ortam
# uv-managed Python next to the venv: the archive then holds everything the environment needs.
export UV_PYTHON_INSTALL_DIR=/content/yue2_python
REQS="/content/yue2_infer-0.1.5-py3-none-any.whl fastapi==0.136.1 uvicorn==0.46.0 torchaudio==2.10.0 librosa
      julius>=0.2.3 dora-search einops pyyaml omegaconf gradio>=5.20,<6 requests"
NODEPS="demucs==4.0.1 openunmix"
KEY=yue2-env-$({ cat /content/yue2_infer-0.1.5-py3-none-any.whl; echo $REQS $NODEPS; } | md5sum | cut -c1-8)
if ! cache_restore $KEY $VENV $UV_PYTHON_INSTALL_DIR; then
    pip install -q uv
    rm -rf $VENV $UV_PYTHON_INSTALL_DIR
    uv venv -q --python 3.12 --python-preference only-managed $VENV
    uv pip install -q --python $VENV/bin/python $REQS
    uv pip install -q --python $VENV/bin/python --no-deps $NODEPS
    cache_save $KEY $VENV $UV_PYTHON_INSTALL_DIR
    cache_prune yue2-env- $KEY
fi
$VENV/bin/python -c "import torch, yue2, demucs.pretrained, librosa, gradio; from transformers import pipeline; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| yue2', yue2.__version__, '| gradio', gradio.__version__, '| Whisper pipeline OK')"
echo "Süre: ${SECONDS} sn"

## 5. Model ağırlıkları (~12 GB)
YuE2 (7.8 GB) ve Demucs (84 MB) cover için; Whisper modelleri (4.4 GB) **✨ Sözleri tanı** için: turbo dili algılar ve Türkçe dışı sözleri yazar, Türkçe model Türkçe sözleri yazar. Model önbelleği açıksa ilk oturumda iner ve Drive'a yazılır (birkaç dk ek); sonraki oturumlarda Drive'dan açılır.

In [ ]:
%%bash
set -e
source /content/cache.sh
PY=/content/yue2_ortam/bin/python
PARTS="/content/models/YuE2-3B /content/models/YuE2-Vae /content/models/whisper-large-v3-turbo
       /content/models/whisper-large-v3-turkish-general /content/models/demucs"
if ! cache_restore models $PARTS; then
$PY - <<'EOF'
from huggingface_hub import snapshot_download
from yue2.storage import MODEL_FILES, MODEL_LICENSES
patterns = sorted(MODEL_FILES) + ["model-?????-of-?????.safetensors"] + ["licenses/" + n for n in sorted(MODEL_LICENSES)]
for repo, folder in (("m-a-p/YuE2-Vae", "/content/models/YuE2-Vae"), ("m-a-p/YuE2-3B", "/content/models/YuE2-3B")):
    print("İndiriliyor:", repo, flush=True)
    snapshot_download(repo, local_dir=folder, allow_patterns=patterns)
# Lyric recognition: multilingual turbo (language detection, non-Turkish) + Turkish fine-tune.
for repo, folder in (("openai/whisper-large-v3-turbo", "/content/models/whisper-large-v3-turbo"),
                     ("turkmedstt/whisper-large-v3-turkish-general", "/content/models/whisper-large-v3-turkish-general")):
    print("İndiriliyor:", repo, flush=True)
    snapshot_download(repo, local_dir=folder, allow_patterns=["*.json", "*.txt", "model.safetensors"])
EOF
mkdir -p /content/models/demucs
cp "$($PY -c 'import demucs, os; print(os.path.dirname(demucs.__file__))')/remote/htdemucs.yaml" /content/models/demucs/
[ -f /content/models/demucs/955717e8-8726e21a.th ] || wget -q -O /content/models/demucs/955717e8-8726e21a.th \
    https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th
cache_save models $PARTS
fi
du -sh /content/models/*
echo "Süre: ${SECONDS} sn"

## 5b. SheetSage2 (~3-5 dk)
Referans şarkının notasını çıkaran, YuE2 ekibinin kendi modeli. yue2'den farklı bir torch sürümü (2.8) istediği için ayrı bir ortama (`/content/sheetsage_ortam`, Python 3.11) kurulur. Model (230 MB) ve dayandığı MERT-v2 (2.5 GB) iner (önbellek açıksa ortamla birlikte yalnızca ilk oturumda). Son satırda **SheetSage2 hazır** görünmeli. Bu hücre hata verirse arayüzde *Nota çıkarıcı: Basit analiz* ile devam edebilirsiniz.

In [ ]:
%%bash
set -e
source /content/cache.sh
# MERT-v2 lives in the Hugging Face cache, not under /content/models (repo = base_model_name_or_path in config.json).
MERT="$(python -c 'from huggingface_hub.constants import HF_HUB_CACHE as c; print(c)')/models--m-a-p--MERT-v2-FullSong"
if ! cache_restore sheetsage /content/models/SheetSage2 "$MERT"; then
python - <<'EOF'
import json
from huggingface_hub import snapshot_download
snapshot_download("m-a-p/SheetSage2", local_dir="/content/models/SheetSage2",
                  ignore_patterns=["render_assets/*", "benchmarks/*", "assets/*"])
# SheetSage2 merges its adapters into this exact MERT-v2 revision (checked by hash at load time).
config = json.load(open("/content/models/SheetSage2/config.json"))
snapshot_download(config["base_model_name_or_path"], revision=config["base_model_revision"])
EOF
cache_save sheetsage /content/models/SheetSage2 "$MERT"
fi
VENV=/content/sheetsage_ortam
export UV_PYTHON_INSTALL_DIR=/content/sheetsage_python
TORCH="torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126"
KEY=sheetsage-env-$({ cat /content/models/SheetSage2/requirements.txt; echo $TORCH; } | md5sum | cut -c1-8)
if ! cache_restore $KEY $VENV $UV_PYTHON_INSTALL_DIR; then
    pip install -q uv
    rm -rf $VENV $UV_PYTHON_INSTALL_DIR
    uv venv -q --python 3.11 --python-preference only-managed $VENV
    uv pip install -q --python $VENV/bin/python $TORCH
    uv pip install -q --python $VENV/bin/python -r /content/models/SheetSage2/requirements.txt
    cache_save $KEY $VENV $UV_PYTHON_INSTALL_DIR
    cache_prune sheetsage-env- $KEY
fi
PY=$VENV/bin/python
$PY -c "import torch, transformers; print('SheetSage2 hazır | torch', torch.__version__, '| CUDA', torch.cuda.is_available())"
echo "Süre: ${SECONDS} sn"

## 6. Model sunucusunu başlat
Colab'da GPU size ayrıldığı için model üretimler arasında bellekte tutulur (`--keep-loaded`).

In [ ]:
import os, subprocess, time, requests
os.makedirs("/content/logs", exist_ok=True)
YUE2_PY = "/content/yue2_ortam/bin/python"  # 4. hücrenin ortamı
server_args = [YUE2_PY, "/content/yue-ui/server/yue2_server.py",
               "--model", "/content/models/YuE2-3B", "--vae", "/content/models/YuE2-Vae",
               "--offline", "--keep-loaded", "--outputs", OUTPUTS, "--budget", str(int(GPU_MEM_GIB)),
               "--sheetsage", "/content/models/SheetSage2", "--sheetsage-python", "/content/sheetsage_ortam/bin/python"]
if GPU_MEM_GIB < 20:
    server_args.append("--offload-ar")  # T4 gibi 16 GB kartlarda bellek tasarrufu
server = subprocess.Popen(server_args, stdout=open("/content/logs/model_server.log", "w"), stderr=subprocess.STDOUT,
                          env={**os.environ, "HF_HUB_OFFLINE": "1"})
for _ in range(90):
    try:
        print(requests.get("http://127.0.0.1:7861/health", timeout=2).json())
        break
    except requests.RequestException:
        time.sleep(2)
else:
    print(open("/content/logs/model_server.log").read()[-3000:])

## 7. Arayüzü başlat
Çıktıda arayüzün linki yazar; arayüz ayrıca bu hücrenin altında da açılır. Link **yalnızca sizin Google hesabınızla** açılır. Adres çubuğunda `localhost` görürseniz Colab'da değil, kendi bilgisayarınızdasınız.

In [ ]:
from google.colab import output
from google.colab.output import eval_js
# Colab's public address for the UI. Gradio builds audio/download links from it; without it the links
# point at an internal host the browser cannot resolve (player stuck at 0:00, download fails).
UI_URL = eval_js("google.colab.kernel.proxyPort(7860)").rstrip("/")
ui = subprocess.Popen([YUE2_PY, "/content/yue-ui/source/yue2_ui.py", "--server_name", "127.0.0.1", "--server_port", "7860"],
                      stdout=open("/content/logs/ui.log", "w"), stderr=subprocess.STDOUT,
                      env={**os.environ, "GRADIO_ANALYTICS_ENABLED": "False", "GRADIO_ROOT_PATH": UI_URL})
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:7860/", timeout=2); break
    except requests.RequestException:
        time.sleep(1)
print("Arayüz linki (yeni sekmede açın):", UI_URL)
output.serve_kernel_port_as_iframe(7860, height=1200)

### 7b. (Yedek) Link açılmazsa
Colab'ın proxy linki çalışmazsa aşağıdaki hücre Gradio'nun **herkese açık** geçici linkini üretir (linki bilen herkes erişebilir; 72 saat geçerli). Linki kimseyle paylaşmayın.

In [ ]:
# ui.terminate()
# ui = subprocess.Popen([YUE2_PY, "/content/yue-ui/source/yue2_ui.py", "--share"],
#                       stdout=open("/content/logs/ui.log", "w"), stderr=subprocess.STDOUT)
# time.sleep(25); print([l for l in open("/content/logs/ui.log") if "gradio.live" in l])

## 8. Loglar (sorun olursa bu çıktıyı gönderin)

In [ ]:
!tail -n 25 /content/logs/model_server.log
!echo ---; tail -n 15 /content/logs/ui.log

### 8b. (Yedek) Son şarkıyı indir
Arayüzdeki indirme çalışmazsa bu hücre en son üretilen şarkıyı doğrudan bilgisayarınıza indirir.

In [ ]:
import glob
from google.colab import files
songs = sorted(glob.glob(f"{OUTPUTS}/*/audio.mp3"), key=os.path.getmtime)
print("\n".join(songs[-5:]) or "Henüz şarkı yok.")
if songs:
    files.download(songs[-1])

## 9. Bitir: GPU'yu bırak
**KAPAT** kutucuğunu işaretleyip bu hücreyi çalıştırın: sunucular kapanır, GPU bırakılır ve compute birimi harcaması durur. Drive kullanmadıysanız `/content` altındaki şarkılar silinir; önce arayüzden indirin.

İlk oturumda önbellek (~25 GB) Drive'a arka planda yüklenir; KAPAT bu yükleme bitene kadar bekler, o yüzden birkaç dakika sürebilir. Bekleyin: yükleme yarım kalırsa sonraki oturumda ağırlıklar yeniden iner.

In [ ]:
KAPAT = False  # @param {type:"boolean"}
# "Tümünü çalıştır" buraya geldiğinde kutucuk işaretsiz olduğu için hiçbir şey kapanmaz.
if KAPAT:
    server.terminate(); ui.terminate()
    if os.path.ismount("/content/drive"):
        # Waits until Drive has uploaded everything written this session (model cache, songs).
        from google.colab import drive
        drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()
else:
    print("Kapatmak için KAPAT kutucuğunu işaretleyip bu hücreyi tekrar çalıştırın.")